In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import preprocessing
import matplotlib.pyplot as plt
%matplotlib inline

import scipy.stats as stats
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import cross_val_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re


from scipy.spatial import cKDTree

In [3]:
dfa=pd.read_csv("enriched_prepared_obs.csv")
dfb=pd.read_csv('enriched_prepared_plant2_obs.csv')
df_pest= pd.read_csv('enriched_prepared_pest_obs.csv')

In [4]:
dfa.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN
0,310657266,2025-08-31 19:37:45,40.799434,-111.012697,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,292.79025,0.000011,288.58110
1,310812803,2025-08-31 14:52:04,40.690640,-110.903167,Flowering,Achillea millefolium,common yarrow,52821,243,27.0,290.41034,0.000012,279.16245
2,310648485,2025-08-31 15:17:00,43.939639,-87.719908,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,294.50253,0.000030,255.68756
3,310647240,2025-08-31 14:42:29,44.741903,-65.519220,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,290.05582,0.000014,265.07320
4,310965925,2025-08-31 08:24:00,38.275494,-120.307077,Flowering,Achillea millefolium,common yarrow,52821,243,9.0,292.76392,0.000010,317.38394
5,310482483,2025-08-31 10:18:21,42.146311,-77.131258,Flowering,Achillea millefolium,common yarrow,52821,243,25.0,293.99230,0.000021,266.84604
6,310494989,2025-08-31 11:52:00,55.955309,-2.999978,Flowering,Achillea millefolium,common yarrow,52821,243,15.0,289.03528,0.000017,219.34627
7,310438565,2025-08-31 11:35:38,52.270790,-2.155858,Flowering,Achillea millefolium,common yarrow,52821,243,15.0,290.30182,0.000007,226.88370
8,310457618,2025-08-31 09:56:00,58.544962,31.377577,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,288.90690,0.000045,192.62874
9,310310316,2025-08-31 08:43:56,-36.915575,174.701256,Flowering,Achillea millefolium,common yarrow,52821,243,15.0,284.50986,0.000045,130.04248


In [5]:
dfb.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN
0,299753119,2025-07-21 16:51:12,-39.434370,176.601988,Fruiting,Melicytus ramiflorus,Māhoe,197063,202,15.0,282.65050,0.000023,118.308100
1,284940738,2025-05-29 08:38:15,-36.863667,174.803194,Flowering,Melicytus ramiflorus,Māhoe,197063,149,15.0,286.51730,0.000076,81.148575
2,281118698,2025-05-11 02:06:12,-43.623842,172.638870,Fruiting,Melicytus ramiflorus,Māhoe,197063,131,15.0,282.71564,0.000025,88.381730
3,280156036,2025-05-11 13:45:12,-40.191279,176.135594,Fruiting,Melicytus ramiflorus,Māhoe,197063,131,15.0,282.61160,0.000026,97.726080
4,273582061,2025-04-26 11:16:37,-36.810763,174.943681,Flowering,Melicytus ramiflorus,Māhoe,197063,116,15.0,288.34723,0.000049,111.342980
5,277585585,2025-04-26 10:05:00,-43.579545,172.633167,Fruiting,Melicytus ramiflorus,Māhoe,197063,116,15.0,282.71564,0.000025,88.381730
6,272136460,2025-04-22 17:13:17,-43.619107,172.650201,Fruiting,Melicytus ramiflorus,Māhoe,197063,112,15.0,282.71564,0.000025,88.381730
7,272034640,2025-04-21 21:10:51,-38.517022,175.582863,Fruiting,Melicytus ramiflorus,Māhoe,197063,111,15.0,283.62088,0.000046,105.719450
8,279300214,2025-04-18 12:45:40,-41.298378,174.010819,Fruiting,Melicytus ramiflorus,Māhoe,197063,108,15.0,282.46506,0.000042,91.797790
9,277671049,2025-04-06 02:15:39,-45.842037,170.660478,Fruiting,Melicytus ramiflorus,Māhoe,197063,96,15.0,285.61716,0.000018,111.994160


In [6]:
df_pest.head(30)

,id,observed_on,latitude,longitude,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN
0,301225953,2025-07-01 08:35:00,41.735947,-121.298850,Acyrthosiphon pisum,Pea Aphid,132882,182,7.0,295.40433,0.000009,340.090900
1,279045138,2025-05-06 09:24:00,42.653488,2.845778,Acyrthosiphon pisum,Pea Aphid,132882,126,7.0,291.44858,0.000020,290.447540
2,272452692,2025-04-23 14:48:51,27.782900,-97.561670,Acyrthosiphon pisum,Pea Aphid,132882,113,14.0,300.76170,0.000029,263.375340
3,219194104,2024-05-29 11:54:36,40.583203,-4.151470,Acyrthosiphon pisum,Pea Aphid,132882,150,8.0,293.12665,0.000017,304.076000
4,217965838,2024-05-24 18:32:00,52.494606,13.326206,Acyrthosiphon pisum,Pea Aphid,132882,145,15.0,291.53833,0.000023,270.828950
5,184813256,2023-09-24 16:13:00,53.558726,-113.540158,Acyrthosiphon pisum,Pea Aphid,132882,267,26.0,278.02536,0.000007,99.595790
6,162263149,2023-05-15 12:40:00,52.557148,13.346573,Acyrthosiphon pisum,Pea Aphid,132882,135,15.0,286.66107,0.000006,274.013950
7,150339950,2023-03-05 12:25:00,46.719042,-117.200319,Acyrthosiphon pisum,Pea Aphid,132882,64,18.0,275.06598,0.000019,141.165770
8,119208095,2022-05-28 15:12:00,52.494056,13.324672,Acyrthosiphon pisum,Pea Aphid,132882,148,15.0,293.75363,0.000014,276.375240
9,118804778,2022-05-26 18:11:35,45.388611,5.361018,Acyrthosiphon pisum,Pea Aphid,132882,146,15.0,293.32890,0.000033,292.538150


In [7]:
# mergeing the two data sets 
plant = pd.concat([dfa,dfb], axis=0, ignore_index=True)
plant.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN
0,310657266,2025-08-31 19:37:45,40.799434,-111.012697,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,292.79025,0.000011,288.58110
1,310812803,2025-08-31 14:52:04,40.690640,-110.903167,Flowering,Achillea millefolium,common yarrow,52821,243,27.0,290.41034,0.000012,279.16245
2,310648485,2025-08-31 15:17:00,43.939639,-87.719908,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,294.50253,0.000030,255.68756
3,310647240,2025-08-31 14:42:29,44.741903,-65.519220,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,290.05582,0.000014,265.07320
4,310965925,2025-08-31 08:24:00,38.275494,-120.307077,Flowering,Achillea millefolium,common yarrow,52821,243,9.0,292.76392,0.000010,317.38394
5,310482483,2025-08-31 10:18:21,42.146311,-77.131258,Flowering,Achillea millefolium,common yarrow,52821,243,25.0,293.99230,0.000021,266.84604
6,310494989,2025-08-31 11:52:00,55.955309,-2.999978,Flowering,Achillea millefolium,common yarrow,52821,243,15.0,289.03528,0.000017,219.34627
7,310438565,2025-08-31 11:35:38,52.270790,-2.155858,Flowering,Achillea millefolium,common yarrow,52821,243,15.0,290.30182,0.000007,226.88370
8,310457618,2025-08-31 09:56:00,58.544962,31.377577,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,288.90690,0.000045,192.62874
9,310310316,2025-08-31 08:43:56,-36.915575,174.701256,Flowering,Achillea millefolium,common yarrow,52821,243,15.0,284.50986,0.000045,130.04248


#### conversion of dates and cleaning of the dates 

In [8]:
# Example: ensure column is string
plant['observed_on'] = plant['observed_on'].astype(str)

# Extract only the date portion using regex
plant['observed_on'] = plant['observed_on'].apply(
    lambda x: re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x).group(0) if re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x) else x
)

# Standardize format (replace '/' with '-')
plant['observed_on'] = plant['observed_on'].str.replace('/', '-', regex=False)


In [9]:
# Example: ensure column is string
df_pest['observed_on'] = df_pest['observed_on'].astype(str)

# Extract only the date portion using regex
df_pest['observed_on'] = df_pest['observed_on'].apply(
    lambda x: re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x).group(0) if re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x) else x
)

# Standardize format (replace '/' with '-')
df_pest['observed_on'] = df_pest['observed_on'].str.replace('/', '-', regex=False)


#### creation of biome_cat which tell about the categories A,B,C,D,E,H for plant data and pest data 


In [10]:

conditions = [
    (plant['biome'].between(1, 5)),
    (plant['biome'].between(6, 8)),
    (plant['biome'].between(9, 14)),
    (plant['biome'].between(15, 21)),
    (plant['biome'].between(22, 24)),
    (plant['biome'] >= 25)
]

choices = ['A', 'B', 'C', 'D', 'E', 'H']

plant['biome_cat'] = np.select(conditions, choices, default='Unknown')


In [11]:

conditions = [
    (df_pest['biome'].between(1, 5)),
    (df_pest['biome'].between(6, 8)),
    (df_pest['biome'].between(9, 14)),
    (df_pest['biome'].between(15, 21)),
    (df_pest['biome'].between(22, 24)),
    (df_pest['biome'] >= 25)
]

choices = ['A', 'B', 'C', 'D', 'E', 'H']

df_pest['biome_cat'] = np.select(conditions, choices, default='Unknown')


#### deletion of the observation which is having less than 600 observations for plant & pest

In [12]:
species_counts = plant['scientific_name'].value_counts()

# 🔹 Step 3: Keep only species with >= 300 observations
valid_species = species_counts[species_counts >= 600].index

# 🔹 Step 4: Filter the dataset
plant = plant[plant['scientific_name'].isin(valid_species)].copy()
plant.head()

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN,biome_cat
0,310657266,2025-08-31,40.799434,-111.012697,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,292.79025,0.000011,288.58110,H
1,310812803,2025-08-31,40.690640,-110.903167,Flowering,Achillea millefolium,common yarrow,52821,243,27.0,290.41034,0.000012,279.16245,H
2,310648485,2025-08-31,43.939639,-87.719908,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,294.50253,0.000030,255.68756,H
3,310647240,2025-08-31,44.741903,-65.519220,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,290.05582,0.000014,265.07320,H
4,310965925,2025-08-31,38.275494,-120.307077,Flowering,Achillea millefolium,common yarrow,52821,243,9.0,292.76392,0.000010,317.38394,C


In [13]:
species_counts = df_pest['scientific_name'].value_counts()

# 🔹 Step 3: Keep only species with >= 300 observations
valid_species = species_counts[species_counts >= 600].index

# 🔹 Step 4: Filter the dataset
df_pest = df_pest[df_pest['scientific_name'].isin(valid_species)].copy()
df_pest.head()

,id,observed_on,latitude,longitude,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN,biome_cat
37,310622040,2025-08-31,39.031541,-85.700086,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,296.12186,0.000013,267.94750,H
38,310860325,2025-08-31,39.962816,-82.772267,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,295.93774,0.000006,278.43228,H
39,310600683,2025-08-31,33.639639,-83.625739,Aedes albopictus,Asian Tiger Mosquito,62984,243,14.0,298.14886,0.000071,250.13303,C
40,310599487,2025-08-31,39.031530,-85.700175,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,296.12186,0.000013,267.94750,H
41,310599519,2025-08-31,39.031552,-85.700217,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,296.12186,0.000013,267.94750,H


#### mapping of pest data with plant data using the temporal data and biome with doy

In [14]:
# Select relevant columns for mapping
pest_map = df_pest[['biome_cat', 'T2M', 'PRECTOTCORR', 'SWGDN', 'scientific_name', 'common_name']]

# Function to map nearest pest to every plant
def map_nearest_pest_no_nan(plant, df_pest):
    merged_list = []

    for biome in plant['biome_cat'].unique():
        plant_b = plant[plant['biome_cat']==biome].copy()
        pest_b = df_pest[df_pest['biome_cat']==biome].copy()

        if pest_b.empty:
            # If no pests in this biome, assign random pest from entire pest dataset
            pest_b = df_pest.copy()

        # Build KDTree on pest environmental features
        tree = cKDTree(pest_b[['T2M','PRECTOTCORR','SWGDN']].values)

        # Query nearest pest for each plant
        _, idxs = tree.query(plant_b[['T2M','PRECTOTCORR','SWGDN']].values)

        # Assign nearest pest info
        plant_b['scientific_name_pest'] = pest_b.iloc[idxs]['scientific_name'].values
        plant_b['common_name_pest'] = pest_b.iloc[idxs]['common_name'].values
        

        merged_list.append(plant_b)

    return pd.concat(merged_list, ignore_index=True)

# Apply mapping
merged_full = map_nearest_pest_no_nan(plant, df_pest)
merged_full.to_csv("plant_with_mapped_pest.csv", index=False)
print(merged_full.head())


          id observed_on   latitude   longitude phenophase  \
0  310657266  2025-08-31  40.799434 -111.012697  Flowering   
1  310812803  2025-08-31  40.690640 -110.903167  Flowering   
2  310648485  2025-08-31  43.939639  -87.719908  Flowering   
3  310647240  2025-08-31  44.741903  -65.519220  Flowering   
4  310482483  2025-08-31  42.146311  -77.131258  Flowering   

        scientific_name    common_name  taxon_id  doy  biome        T2M  \
0  Achillea millefolium  common yarrow     52821  243   26.0  292.79025   
1  Achillea millefolium  common yarrow     52821  243   27.0  290.41034   
2  Achillea millefolium  common yarrow     52821  243   26.0  294.50253   
3  Achillea millefolium  common yarrow     52821  243   26.0  290.05582   
4  Achillea millefolium  common yarrow     52821  243   25.0  293.99230   

   PRECTOTCORR      SWGDN biome_cat       scientific_name_pest  \
0     0.000011  288.58110         H             Empoasca fabae   
1     0.000012  279.16245         H         

In [15]:
merged_full.head()

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN,biome_cat,scientific_name_pest,common_name_pest
0,310657266,2025-08-31,40.799434,-111.012697,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,292.79025,0.000011,288.58110,H,Empoasca fabae,Potato Leafhopper
1,310812803,2025-08-31,40.690640,-110.903167,Flowering,Achillea millefolium,common yarrow,52821,243,27.0,290.41034,0.000012,279.16245,H,Ostrinia nubilalis,European Corn Borer Moth
2,310648485,2025-08-31,43.939639,-87.719908,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,294.50253,0.000030,255.68756,H,Popillia japonica,Japanese Beetle
3,310647240,2025-08-31,44.741903,-65.519220,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,290.05582,0.000014,265.07320,H,Lymantria dispar,Spongy Moth
4,310482483,2025-08-31,42.146311,-77.131258,Flowering,Achillea millefolium,common yarrow,52821,243,25.0,293.99230,0.000021,266.84604,H,Leptinotarsa decemlineata,Colorado Potato Beetle


In [16]:
merged_full.to_csv('plant_pest_dataset.csv',index=False)

In [17]:
merged_full.drop(columns=['id','latitude','longitude','taxon_id','biome'],inplace=True)

In [18]:
#Add cyclic encoding for DOY (so model understands seasonality)
merged_full['doy_sin'] = np.sin(2 * np.pi * merged_full['doy'] / 365)
merged_full['doy_cos'] = np.cos(2 * np.pi * merged_full['doy'] / 365)


In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# ----------------------------
# Encode categorical features
# ----------------------------
enc_biome = LabelEncoder()
merged_full['biome_enc'] = enc_biome.fit_transform(merged_full['biome_cat'])

# Encode targets
y_cols = ['scientific_name', 'common_name', 'scientific_name_pest', 'common_name_pest', 'phenophase']
label_encoders = {}
for col in y_cols:
    le = LabelEncoder()
    merged_full[col + '_enc'] = le.fit_transform(merged_full[col])
    label_encoders[col] = le

# Encode scientific_name for phenophase feature
enc_species = LabelEncoder()
merged_full['species_enc'] = enc_species.fit_transform(merged_full['scientific_name'])

# ----------------------------
# Features
# ----------------------------
features = ['biome_enc', 'doy', 'T2M', 'PRECTOTCORR', 'SWGDN']

# ----------------------------
# Train RandomForest per target
# ----------------------------
models = {}

# 1️⃣ Species, common_name, pests
for col in ['scientific_name', 'common_name', 'scientific_name_pest', 'common_name_pest']:
    target_enc = col + '_enc'
    X_train, X_test, y_train, y_test = train_test_split(
        merged_full[features], merged_full[target_enc], test_size=0.2, random_state=42
    )
    rf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=3, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    models[col] = rf

# 2️⃣ Phenophase (include species as feature)
X_phase = merged_full[features + ['species_enc']]
y_phase = merged_full['phenophase_enc']
X_train, X_test, y_train, y_test = train_test_split(X_phase, y_phase, test_size=0.2, random_state=42)
rf_phase = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_phase.fit(X_train, y_train)
models['phenophase'] = rf_phase

# ----------------------------
# Prediction function
# ----------------------------
def get_top10_plants_with_pests(doy, T2M, PRECTOTCORR, SWGDN, biome_cat):
    biome_enc = enc_biome.transform([biome_cat])[0]
    df_input = pd.DataFrame([[biome_enc, doy, T2M, PRECTOTCORR, SWGDN]], columns=features)
    
    # 1️⃣ Top 10 plant species
    species_model = models['scientific_name']
    probs = species_model.predict_proba(df_input)[0]
    classes = label_encoders['scientific_name'].inverse_transform(np.arange(len(probs)))
    top_idx = np.argsort(probs)[::-1][:10]
    
    results = []
    for idx in top_idx:
        plant_name = classes[idx]
        common_name = merged_full.loc[merged_full['scientific_name'] == plant_name, 'common_name'].mode()[0]
        
        # Phenophase prediction for this plant
        plant_code = enc_species.transform([plant_name])[0]
        phase_model = models['phenophase']
        phase_code = phase_model.predict([[biome_enc, doy, T2M, PRECTOTCORR, SWGDN, plant_code]])[0]
        phase_name = label_encoders['phenophase'].inverse_transform([phase_code])[0]
        
        # Top pest for this plant
        pest_model = models['scientific_name_pest']
        pest_probs = pest_model.predict_proba(df_input)[0]
        pest_classes = label_encoders['scientific_name_pest'].inverse_transform(np.arange(len(pest_probs)))
        top_pest_idx = np.argmax(pest_probs)
        pest_name = pest_classes[top_pest_idx]
        pest_common = merged_full.loc[merged_full['scientific_name_pest'] == pest_name, 'common_name_pest'].mode()[0]
        
        results.append({
            'scientific_name': plant_name,
            'common_name': common_name,
            'phenophase': phase_name,
            'scientific_name_pest': pest_name,
            'common_name_pest': pest_common
        })
    
    return results

# ----------------------------
# Example usage
# ----------------------------


In [34]:
top10 = get_top10_plants_with_pests(
    doy=243,
    T2M=292.79,
    PRECTOTCORR=0.000011,
    SWGDN=288.58,
    biome_cat='H'
)

for plant in top10:
    print(plant)


C:\Users\aayus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\aayus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\aayus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\aayus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-pac

{'scientific_name': 'Chamaenerion angustifolium', 'common_name': 'fireweed', 'phenophase': 'Flowering', 'scientific_name_pest': 'Leptinotarsa decemlineata', 'common_name_pest': 'Colorado Potato Beetle'}
{'scientific_name': 'Impatiens capensis', 'common_name': 'common jewelweed', 'phenophase': 'Flowering', 'scientific_name_pest': 'Leptinotarsa decemlineata', 'common_name_pest': 'Colorado Potato Beetle'}
{'scientific_name': 'Anaphalis margaritacea', 'common_name': 'pearly everlasting', 'phenophase': 'Flowering', 'scientific_name_pest': 'Leptinotarsa decemlineata', 'common_name_pest': 'Colorado Potato Beetle'}
{'scientific_name': 'Ageratina altissima', 'common_name': 'white snakeroot', 'phenophase': 'Flowering', 'scientific_name_pest': 'Leptinotarsa decemlineata', 'common_name_pest': 'Colorado Potato Beetle'}
{'scientific_name': 'Lythrum salicaria', 'common_name': 'purple loosestrife', 'phenophase': 'Flowering', 'scientific_name_pest': 'Leptinotarsa decemlineata', 'common_name_pest': 'Col

In [20]:
df_pest.head()

,id,observed_on,latitude,longitude,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN,biome_cat
37,310622040,2025-08-31,39.031541,-85.700086,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,296.12186,0.000013,267.94750,H
38,310860325,2025-08-31,39.962816,-82.772267,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,295.93774,0.000006,278.43228,H
39,310600683,2025-08-31,33.639639,-83.625739,Aedes albopictus,Asian Tiger Mosquito,62984,243,14.0,298.14886,0.000071,250.13303,C
40,310599487,2025-08-31,39.031530,-85.700175,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,296.12186,0.000013,267.94750,H
41,310599519,2025-08-31,39.031552,-85.700217,Aedes albopictus,Asian Tiger Mosquito,62984,243,25.0,296.12186,0.000013,267.94750,H
